# Notebook 1 — Pipeline de traitement

Ce notebook **rejoue le pipeline `frost_days/*.py` étape par étape** pour rendre lisible chaque transformation :

1. **Référentiel communes** → lat/lon de la commune cible
2. **Téléchargement Météo-France** → CSV départemental (cache local)
3. **Filtrage temporel** au chargement (chunks, pour ne pas exploser la RAM sur >100 M lignes)
4. **Liste des stations** du département
5. **Haversine** → N stations les plus proches de la commune
6. **Sélection station valide** (≤ 35 % de NaN sur TN)
7. **Calcul jours de gel** (TN ≤ 0 °C) + agrégations par année et jour de l'année

> Cible : **Dijon (21)** sur **2014-01-01 → 2023-12-31**.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np

COMMUNE = 'Dijon'
DEPT    = '21'
START   = '2014-01-01'
END     = '2023-12-31'

### Stack technique — pourquoi ces outils (et pas d'autres)

| Outil | Rôle dans le pipeline | Pourquoi celui-là |
|---|---|---|
| **pandas** | lecture du CSV `.gz`, **lecture par chunks** (`chunksize`), filtrage, `groupby` | tout est tabulaire ; pandas gère nativement le gzip et permet de filtrer la plage de dates *au fil de l'eau* (jamais tout en RAM) |
| **numpy** | **Haversine vectorisé** (sin/cos sur des tableaux), régressions `polyfit` | distance commune → *toutes* les stations en une opération, sans boucle Python |
| **requests** | téléchargement data.gouv + **cache local** | on télécharge une fois, on relit ensuite depuis le disque |
| stdlib (`pathlib`, `argparse`) | chemins, CLI | zéro dépendance superflue |

**Choix d'architecture assumés :**
- **Pas de GeoPandas** : on n'a besoin que d'une distance point→points, le Haversine maison (≈ 8 lignes) suffit — inutile d'embarquer des géométries.
- **Pas de base de données** : un cache `.gz` + lecture filtrée couvre la volumétrie (un département à la fois). 
- **Pas de scikit-learn** : les tendances de l'app sont des régressions linéaires à 1 variable → `numpy.polyfit` est largement suffisant et transparent.

## 1. Référentiel communes

Source : `data.gouv.fr` (CSV ~21 Mo). On le télécharge une fois et on le met en cache local sous `data/communes/`. Le séparateur peut varier selon la version (`,` ou `;`), donc on laisse pandas le détecter.

In [2]:
from frost_days.communes import download_communes, load_communes, find_commune

path = download_communes()
print('Fichier :', path)

communes = load_communes()
print(f'{len(communes):,} communes chargées')
communes.head()

Fichier : C:\Users\alexi\Desktop\Frost_days\data\communes\communes-france.csv
34,868 communes chargées


,nom,dept,lat,lon,nom_norm
0,L'Abergement-Clémenciat,01,46.151,4.921,l'abergement-clémenciat
1,L'Abergement-de-Varey,01,46.007,5.423,l'abergement-de-varey
2,Ambérieu-en-Bugey,01,45.958,5.360,ambérieu-en-bugey
3,Ambérieux-en-Dombes,01,45.996,4.903,ambérieux-en-dombes
4,Ambléon,01,45.748,5.601,ambléon


In [3]:
# Lookup commune -> (lat, lon). Le module applique aussi le dict de correctifs
# pour les communes connues sans coordonnées dans le référentiel (Paris, Marseille, ...).
lat, lon = find_commune(communes, COMMUNE, DEPT)
print(f'{COMMUNE} ({DEPT}) -> lat={lat:.4f}, lon={lon:.4f}')

# Diagnostic : combien de communes du référentiel sont sans coord ?
missing = communes['lat'].isna().sum()
print(f'Communes sans coord : {missing} / {len(communes)} ({missing/len(communes):.2%})')

Dijon (21) -> lat=47.3220, lon=5.0420
Communes sans coord : 5 / 34868 (0.01%)


## 2. Téléchargement du département cible

Le pattern d'URL Météo-France découvert via l'API data.gouv :

```
https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/BASE/QUOT/Q_{DEPT}_previous-1950-2024_RR-T-Vent.csv.gz
```

Un fichier départemental fait ~10-30 Mo compressé (~250-700 Mo décompressé), donc on stocke le `.gz` et on lit en streaming.

### Pourquoi `usecols` — on ne lit que 7 colonnes sur 60

Le fichier brut compte **60 colonnes** ; on en désérialise **7** dès la lecture (`usecols=`). Ce n'est pas arbitraire, chaque colonne répond à un mot de la question *« compter les jours où `TN ≤ 0` à la station la plus proche d'une commune, sur une période »* :

| Colonne | À quoi elle sert *concrètement* | Vital ? |
|---|---|---|
| `TN` | « TN ≤ 0 » → **LA** variable du gel | oui |
| `AAAAMMJJ` | « sur une période », agrégats par an / par jour | oui |
| `NUM_POSTE` | « à la station » → clé de groupement, choix d'une station | oui |
| `LAT`, `LON` | « la plus proche » → distance Haversine | oui |
| `ALTI` | analyse **altitude × gel** de l'app | bonus assumé |
| `NOM_USUEL` | affichage lisible (« Dijon-Toison » plutôt que `21231007`) | confort |

> ⚠️ La sortie ci-dessus peut afficher 6 colonnes si elle a été exécutée avant l'ajout d'`ALTI` ; la version courante de `USECOLS` en contient **7**.

**Les 53 autres sont écartées pour 2 raisons seulement** (détail chiffré : notebook 2, § 1 bis) :
**(1) mauvaise mesure / hors-sujet** — `TX` (max), `TM` (moyenne), pluie, vent ; **(2) trop lacunaire** — `TNSOL` 96 % de vide, `DG` 88 %. Sans compter que **la moitié du fichier** n'est que des **flags qualité `Q…`** (métadonnée). Lire 7 colonnes au lieu de 60 = **moins de RAM et lecture plus rapide** — un vrai gain, pas juste de la propreté.

In [4]:
from frost_days.data_loader import download_dept, USECOLS

fpath = download_dept(DEPT)
print('Fichier :', fpath, f'({fpath.stat().st_size/1e6:.1f} Mo)')
print('Colonnes lues :', USECOLS)

Fichier : C:\Users\alexi\Desktop\Frost_days\data\meteo\Q_21_previous-1950-2024_RR-T-Vent.csv.gz (12.0 Mo)
Colonnes lues : ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'AAAAMMJJ', 'TN']


## 3. Chargement filtré (chunks)

`load_dept` lit le `.csv.gz` par **chunks de 500 000 lignes** et filtre la fenêtre temporelle dans la foulée. C'est la clé pour rester sous le Go de RAM même sur les gros départements. Les colonnes inutiles (vent, précipitations…) ne sont pas chargées (`usecols=`).

In [5]:
from frost_days.data_loader import load_dept, list_stations

df = load_dept(DEPT, start=START, end=END)
print(f'{len(df):,} lignes (toutes stations confondues) sur la plage {START} -> {END}')
df.head()

146,100 lignes (toutes stations confondues) sur la plage 2014-01-01 -> 2023-12-31


,NUM_POSTE,NOM_USUEL,LAT,LON,AAAAMMJJ,TN,date
0,21010001,ALOXE CORTON,47.069,4.8615,20140101,NaN,2014-01-01
1,21010001,ALOXE CORTON,47.069,4.8615,20140102,NaN,2014-01-02
2,21010001,ALOXE CORTON,47.069,4.8615,20140103,NaN,2014-01-03
3,21010001,ALOXE CORTON,47.069,4.8615,20140104,NaN,2014-01-04
4,21010001,ALOXE CORTON,47.069,4.8615,20140105,NaN,2014-01-05


In [6]:
stations = list_stations(df)
print(f'{len(stations)} stations dans le département {DEPT}')
stations.head()

58 stations dans le département 21


,NUM_POSTE,NOM_USUEL,LAT,LON
0,21010001,ALOXE CORTON,47.069000,4.861500
1,21012001,AMPILLY LE SEC,47.808333,4.529000
2,21043001,BAIGNEUX LES JUIFS,47.600333,4.645167
3,21056001,BEIRE LE CHATEL,47.413833,5.208333
4,21065001,BESSEY,47.086000,4.743167


**Observation —** les 5 stations les plus proches de Dijon s'échelonnent de **2 km** (`DIJON`) à **15 km** (`VARANGES`). À ce stade on ne classe que par **distance** : rien ne dit encore qu'une station proche possède réellement des relevés de température. C'est tout l'enjeu de l'étape suivante (filtre qualité).

## 4-5. Haversine + sélection des stations proches

Distance grand-cercle (km), vectorisée NumPy. On garde les 5 plus proches.

**Observation & hypothèse —** la station la plus proche, `DIJON` (2 km), a **65 % de `TN` manquantes** → **rejetée**. `DIJON TOISON` (4 km, 11 % de NaN) est la première à passer le seuil → **retenue**. `VARANGES` est à 100 % de NaN (poste sans thermomètre). **Hypothèse :** *proximité* et *fiabilité* sont décorrélées — une stratégie « plus proche absolue » se tromperait souvent. La règle **top-5 puis seuil 35 %** est donc nécessaire, pas cosmétique (c'est la justification métier du choix de station).

In [7]:
from frost_days.stations import nearest_stations

top5 = nearest_stations(stations, lat, lon, top_n=5)
top5

,NUM_POSTE,NOM_USUEL,LAT,LON,dist_km
0,21231001,DIJON,47.319167,5.014333,2.109159
1,21231007,DIJON TOISON,47.358333,5.043333,4.041299
2,21473001,DIJON-LONGVIC,47.267833,5.088333,6.963284
3,21390001,MARSANNAY LA COTE,47.266667,4.986833,7.427382
4,21656001,VARANGES,47.233333,5.196667,15.275794


## 6. Validation des stations — règle des 35 % de NaN

On itère de la plus proche à la plus lointaine, on prend la première dont le taux de NaN sur `TN` ≤ 35 %.

**Observation —** le gel annuel oscille entre **38 et 59 jours** sans tendance monotone, et **2018 est absente** du tableau (aucune observation valide cette année-là à `DIJON TOISON`). On voit aussi que **2019** ne compte que 334 jours sur 365 (`total_days`). **Hypothèse :** sur 10 ans la variabilité naturelle masque tout signal climatique ; et les années à couverture partielle sous-estiment un peu le compte → pour une vraie tendance, il faudrait **normaliser** (`frost_days × 365 / total_days`) et viser ≥ 30 ans.

**Observation & hypothèse —** les jours les plus gélifs sont **tous en cœur d'hiver** : 24/01 (7/8 ans), 10/12 (7/9), puis une grappe fin janvier / mi-février. **Hypothèse :** ce ne sont pas des artefacts aléatoires, ils tombent au creux thermique de l'hiver continental bourguignon. **Limite à garder en tête :** l'échantillon par jour est **≤ 10** → une fréquence « 7/8 = 88 % » a un intervalle de confiance large ; on la lit comme une *tendance de risque*, pas comme une probabilité exacte. C'est précisément cette colonne `freq` que l'app trace pour le calendrier de risque de gel.

In [8]:
expected = (pd.Timestamp(END) - pd.Timestamp(START)).days + 1
diag = []
for _, row in top5.iterrows():
    sub = df[df['NUM_POSTE'] == row['NUM_POSTE']]
    n_obs = sub['TN'].notna().sum()
    diag.append({
        'NUM_POSTE': row['NUM_POSTE'],
        'NOM_USUEL': row['NOM_USUEL'],
        'dist_km':   row['dist_km'],
        'n_obs_TN':  n_obs,
        'missing_ratio': 1 - n_obs/expected,
        'valide_35%': (1 - n_obs/expected) <= 0.35,
    })
pd.DataFrame(diag)

,NUM_POSTE,NOM_USUEL,dist_km,n_obs_TN,missing_ratio,valide_35%
0,21231001,DIJON,2.109159,1279,0.649781,False
1,21231007,DIJON TOISON,4.041299,3256,0.108434,True
2,21473001,DIJON-LONGVIC,6.963284,3652,0.000000,True
3,21390001,MARSANNAY LA COTE,7.427382,2677,0.266977,True
4,21656001,VARANGES,15.275794,0,1.000000,False


## 7. Calcul des jours de gel

Pipeline complet encapsulé dans `compute_frost_days`.

In [9]:
from frost_days.frost import compute_frost_days

res = compute_frost_days(COMMUNE, DEPT, lat, lon, START, END)
print(f'Station retenue : {res.station_name} [{res.station_id}]')
print(f'  distance     : {res.station_distance_km:.1f} km')
print(f'  NaN sur TN   : {res.missing_ratio:.1%}')
print(f'Jours de gel total          : {res.frost_days_total}')
print(f'Jours de gel moyen / annee  : {res.frost_days_per_year_mean:.1f}')

Station retenue : DIJON TOISON [21231007]
  distance     : 4.0 km
  NaN sur TN   : 10.8%
Jours de gel total          : 439
Jours de gel moyen / annee  : 48.8


In [10]:
res.per_year

,year,frost_days,total_days
0,2014,41,365
1,2015,48,365
2,2016,49,366
3,2017,59,365
4,2019,38,334
5,2020,45,366
6,2021,58,365
7,2022,53,365
8,2023,48,365


In [11]:
res.per_day_of_year.sort_values('freq', ascending=False).head(15)

,month,day,n_frost,n_obs,freq
23,1,24,7,8,0.875000
343,12,10,7,9,0.777778
24,1,25,6,8,0.750000
21,1,22,6,8,0.750000
20,1,21,6,8,0.750000
18,1,19,6,8,0.750000
43,2,13,6,9,0.666667
336,12,3,6,9,0.666667
42,2,12,6,9,0.666667
25,1,26,5,8,0.625000


---
**Sortie du pipeline :** un objet `FrostResult` exposant `frost_days_total`, `frost_days_per_year_mean`, `per_year` (DataFrame), `per_day_of_year` (DataFrame), ainsi que les métadonnées de la station retenue. C'est ce que `app.py` consomme pour produire les graphiques Streamlit.